In [1]:
# Merge per-video CLIP .npy → one gallery matrix (+ row map for later FAISS / search).
# Same merge idea as engine.Embedder.load_kf_embedder.
from pathlib import Path

# Point at features/clip (Lxx/VIDEO_ID.npy). Repo working set or media full index.
# CLIP_DIR = Path(r"d:/HCMUS_ComputerScience/code/AIC2026/features/clip")
CLIP_DIR = Path(r"C:/AIC2026-media/features/clip")

OUT_NPY = CLIP_DIR / "embeddings.npy"
OUT_MAP = CLIP_DIR / "gallery_map.csv"  # video_id, start_row, n_rows
OVERWRITE = True  # True = rebuild even if embeddings.npy exists

print("CLIP_DIR =", CLIP_DIR.resolve())
print("OUT_NPY  =", OUT_NPY)
print("OUT_MAP  =", OUT_MAP)


CLIP_DIR = C:\AIC2026-media\features\clip
OUT_NPY  = C:\AIC2026-media\features\clip\embeddings.npy
OUT_MAP  = C:\AIC2026-media\features\clip\gallery_map.csv


In [2]:
import csv

import numpy as np

if not CLIP_DIR.is_dir():
    raise SystemExit(f"Not a directory: {CLIP_DIR}")

npy_paths = sorted(CLIP_DIR.glob("*/*.npy"))
if not npy_paths:
    npy_paths = sorted(CLIP_DIR.glob("*.npy"))
npy_paths = [p for p in npy_paths if p.name != "embeddings.npy"]
if not npy_paths:
    raise SystemExit(f"No per-video .npy under {CLIP_DIR}")

print(f"videos={len(npy_paths)}")

if OUT_NPY.is_file() and OUT_MAP.is_file() and not OVERWRITE:
    emb = np.load(OUT_NPY)
    print(f"Skip merge (exists). embeddings.npy shape={emb.shape} dtype={emb.dtype}")
    print(f"gallery_map.csv exists: {OUT_MAP}")
else:
    blocks = []
    rows = []
    start = 0
    for p in npy_paths:
        block = np.load(p).astype(np.float32, copy=False)
        if block.ndim != 2:
            raise SystemExit(f"Expected 2D {p}, got {block.shape}")
        n = int(block.shape[0])
        blocks.append(block)
        rows.append({"video_id": p.stem, "start_row": start, "n_rows": n})
        start += n
        print(f"  {p.parent.name}/{p.name}  shape={block.shape}", flush=True)

    emb = np.concatenate(blocks, axis=0)
    np.save(OUT_NPY, emb)
    with OUT_MAP.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["video_id", "start_row", "n_rows"])
        w.writeheader()
        w.writerows(rows)

    print(f"Wrote {OUT_NPY}  shape={emb.shape} dtype={emb.dtype}")
    print(f"Wrote {OUT_MAP}  ({len(rows)} videos, total_rows={start})")


videos=873
  L21/L21_V001.npy  shape=(500, 512)
  L21/L21_V002.npy  shape=(516, 512)
  L21/L21_V003.npy  shape=(503, 512)
  L21/L21_V005.npy  shape=(380, 512)
  L21/L21_V006.npy  shape=(453, 512)
  L21/L21_V007.npy  shape=(386, 512)
  L21/L21_V008.npy  shape=(549, 512)
  L21/L21_V009.npy  shape=(506, 512)
  L21/L21_V010.npy  shape=(461, 512)
  L21/L21_V011.npy  shape=(487, 512)
  L21/L21_V012.npy  shape=(418, 512)
  L21/L21_V013.npy  shape=(537, 512)
  L21/L21_V014.npy  shape=(523, 512)
  L21/L21_V015.npy  shape=(630, 512)
  L21/L21_V016.npy  shape=(495, 512)
  L21/L21_V017.npy  shape=(409, 512)
  L21/L21_V018.npy  shape=(515, 512)
  L21/L21_V019.npy  shape=(496, 512)
  L21/L21_V021.npy  shape=(393, 512)
  L21/L21_V022.npy  shape=(366, 512)
  L21/L21_V023.npy  shape=(512, 512)
  L21/L21_V024.npy  shape=(533, 512)
  L21/L21_V025.npy  shape=(524, 512)
  L21/L21_V026.npy  shape=(534, 512)
  L21/L21_V027.npy  shape=(558, 512)
  L21/L21_V028.npy  shape=(406, 512)
  L21/L21_V029.npy  shape=(

In [3]:
# Quick sanity: dim should be 512 for ViT-B-32; map rows cover full matrix.
import csv

emb = np.load(OUT_NPY)
with OUT_MAP.open(encoding="utf-8", newline="") as f:
    map_rows = list(csv.DictReader(f))

covered = sum(int(r["n_rows"]) for r in map_rows)
print("shape =", emb.shape)
print("map videos =", len(map_rows), " covered rows =", covered)
assert covered == emb.shape[0], "gallery_map row count != embeddings rows"
assert emb.shape[-1] == 512, f"expected dim 512, got {emb.shape[-1]}"
print("OK")


shape = (146378, 512)
map videos = 873  covered rows = 146378
OK


In [4]:
! pip install faiss-cpu


In [5]:
import numpy as np
import faiss


all_clips = np.load(CLIP_DIR / "embeddings.npy")
all_clips = all_clips.astype(np.float32)

assert all_clips.ndim == 2

dimension = all_clips.shape[1]
my_index = faiss.IndexFlatIP(dimension)

my_index.add(all_clips)

faiss.write_index(my_index, str(CLIP_DIR / "index.faiss"))


